# PCG-to-ECG Signal Reconstruction: Preprocessing & Augmentation

**Ablation Study Code** for: *PCG-to-ECG Signal Reconstruction Using Conditional GAN with Multi-Scale Attention* (EUSIPCO 2026)

**Authors:** Zafar Ahmad, Nazia Perwaiz, Muhammad Salman Khan

---

This notebook contains:
1. **Polarity correction** (Trail-4) — doubled correlation from 0.30 to 0.61
2. **Z-score normalization** — applied across all trails
3. **Data augmentation pipeline** (Trail-2 onward) — heart-rate stretch, baseline wander, bandpass noise

## Imports

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
import scipy.signal as signal

---
## Polarity Correction (Trail-4)

**Problem:** The EPHNOGRAM dataset does not enforce consistent ECG polarity. About half the recordings have R-peaks pointing downward. After z-score normalization, positive and negative R-peaks average out, halving correlation.

**Fix:** If `|min(ecg)| > |max(ecg)|`, flip the signal.

**Impact (Trail-3 → Trail-4, same model weights):**
- Pearson correlation: 0.30 → 0.61 (+0.31)
- PSNR: 14 dB → 17.1 dB (+3.1 dB)
- R-peak error: 0.52 → 0.126 (-76%)

In [ ]:
def ensure_positive_polarity(ecg):
    """
    Flip the ECG so that R-peaks point upward (positive).

    If |min(ecg)| > |max(ecg)| the signal is inverted
    (R-peaks pointing down). Multiplying by -1 restores
    standard polarity.

    Parameters
    ----------
    ecg : np.ndarray, shape (T,)
        Raw ECG signal (any normalization).

    Returns
    -------
    ecg : np.ndarray, shape (T,)
        Polarity-corrected signal.
    """
    if np.abs(np.min(ecg)) > np.abs(np.max(ecg)):
        ecg = -ecg
    return ecg

### Quick demo: polarity correction

In [ ]:
# Simulated inverted ECG (R-peak at -3.5, baseline near 0)
t = np.linspace(0, 1, 8000)
ecg_inverted = -3.5 * np.exp(-((t - 0.5)**2) / 0.0001)  # downward spike

print(f"Before: min={ecg_inverted.min():.2f}, max={ecg_inverted.max():.2f}")
ecg_fixed = ensure_positive_polarity(ecg_inverted)
print(f"After:  min={ecg_fixed.min():.2f}, max={ecg_fixed.max():.2f}")

---
## Z-Score Normalization

In [ ]:
def normalize_zscore(sig):
    """
    Per-segment z-score normalization: (x - mu) / sigma.

    All trails apply this step. The resulting amplitude range
    spans roughly [-3, +3] for baseline and up to +/- 4-5 at
    R-peaks. This range is wider than [0, 1] normalization,
    which inflates RMSE (see paper Section IV-D, Limitation 4).

    Parameters
    ----------
    sig : np.ndarray, shape (T,)

    Returns
    -------
    np.ndarray, shape (T,)
    """
    mu = sig.mean()
    std = sig.std() + 1e-8
    return (sig - mu) / std

---
## Data Augmentation (Trail-2 onward)

Four augmentation types, applied stochastically per sample during training:

| Augmentation | Probability | Range |
|-------------|-------------|-------|
| Amplitude scaling | 50% | 0.9–1.1x |
| Heart-rate stretch | 60% | 0.85–1.15x |
| Baseline wander | 30% | 0.1–1.0 Hz |
| Bandpass noise | 20% | SNR 25–40 dB |

In [ ]:
def augment_heart_rate(ecg, pcg, scale_factor, fs=8000):
    """
    Time-stretch both signals to simulate heart-rate variation.

    A scale_factor of 1.15 compresses the waveform (faster
    heart rate); 0.85 stretches it (slower heart rate).
    The output is padded or cropped to the original length.
    """
    T = len(ecg)
    new_len = int(T / scale_factor)
    indices = np.linspace(0, T - 1, new_len)

    ecg_s = interp1d(np.arange(T), ecg, kind='cubic',
                      fill_value='extrapolate')(indices)
    pcg_s = interp1d(np.arange(T), pcg, kind='cubic',
                      fill_value='extrapolate')(indices)

    if new_len > T:
        ecg_s, pcg_s = ecg_s[:T], pcg_s[:T]
    else:
        ecg_s = np.pad(ecg_s, (0, T - new_len), mode='edge')
        pcg_s = np.pad(pcg_s, (0, T - new_len), mode='edge')

    return ecg_s.astype(np.float32), pcg_s.astype(np.float32)

In [ ]:
def add_baseline_wander(ecg, pcg, frequency=None, amplitude=0.1, fs=8000):
    """
    Add low-frequency sinusoidal drift (0.1-1 Hz) simulating
    respiration or electrode movement.
    """
    if frequency is None:
        frequency = np.random.uniform(0.1, 1.0)
    t = np.arange(len(ecg), dtype=np.float32) / fs
    wander = amplitude * np.sin(2 * np.pi * frequency * t)
    return (ecg + wander).astype(np.float32), \
           (pcg + wander * 0.3).astype(np.float32)

In [ ]:
def add_bandpass_noise(ecg, pcg, snr_db=None, fs=8000):
    """
    Add Gaussian noise band-limited to 20-200 Hz
    (the cardiac-relevant frequency band).
    """
    if snr_db is None:
        snr_db = np.random.uniform(25, 40)

    sos = signal.butter(4, [20, 200], btype='bandpass', fs=fs, output='sos')

    for sig, power_ref in [(ecg, np.mean(ecg**2)),
                            (pcg, np.mean(pcg**2))]:
        noise = np.random.normal(0, 1, len(sig)).astype(np.float32)
        noise = signal.sosfilt(sos, noise).astype(np.float32)
        target_power = power_ref / (10 ** (snr_db / 10))
        noise *= np.sqrt(target_power / (np.mean(noise**2) + 1e-12))
        sig += noise

    return ecg, pcg

In [ ]:
def augment(ecg, pcg, epoch=1, fs=8000):
    """
    Combined augmentation pipeline applied per sample
    during training.

    Steps (stochastic):
        1. Amplitude scaling  (p=0.5, range 0.9-1.1)
        2. Heart-rate stretch (p=0.6, factor 0.85-1.15)
        3. Baseline wander    (p=0.3, 0.1-1.0 Hz)
        4. Band-limited noise (p=0.2, SNR 25-40 dB)
    """
    ecg = ecg.copy().astype(np.float32)
    pcg = pcg.copy().astype(np.float32)

    if np.random.rand() < 0.5:
        s = np.random.uniform(0.9, 1.1)
        ecg *= s
        pcg *= s

    if np.random.rand() < 0.6:
        ecg, pcg = augment_heart_rate(
            ecg, pcg, np.random.uniform(0.85, 1.15), fs)

    if np.random.rand() < 0.3:
        ecg, pcg = add_baseline_wander(ecg, pcg, fs=fs)

    if np.random.rand() < 0.2:
        ecg, pcg = add_bandpass_noise(ecg, pcg, fs=fs)

    return ecg, pcg

---
## Full Preprocessing Pipeline

The order matters:
1. **Polarity correction** (before normalization)
2. **Augmentation** (on raw-polarity-fixed signals)
3. **Z-score normalization** (last step)

In [ ]:
def preprocess(ecg, pcg, do_augment=False, fix_polarity=True, epoch=1):
    """
    Full preprocessing pipeline.

    Parameters
    ----------
    ecg, pcg : np.ndarray, shape (T,)
    do_augment : bool, apply augmentation (training only)
    fix_polarity : bool, apply polarity correction (Trail-4)
    epoch : int, current training epoch

    Returns
    -------
    ecg, pcg : np.ndarray, shape (T,), preprocessed
    """
    # Step 1: Polarity correction (before normalization)
    if fix_polarity:
        ecg = ensure_positive_polarity(ecg)

    # Step 2: Augmentation (training only)
    if do_augment:
        ecg, pcg = augment(ecg, pcg, epoch=epoch)

    # Step 3: Z-score normalization (last)
    ecg = normalize_zscore(ecg)
    pcg = normalize_zscore(pcg)

    return ecg.astype(np.float32), pcg.astype(np.float32)

# Quick test
dummy_ecg = np.random.randn(64000).astype(np.float32)
dummy_pcg = np.random.randn(64000).astype(np.float32)
ecg_out, pcg_out = preprocess(dummy_ecg, dummy_pcg, do_augment=True)
print(f"ECG: mean={ecg_out.mean():.4f}, std={ecg_out.std():.4f}, shape={ecg_out.shape}")
print(f"PCG: mean={pcg_out.mean():.4f}, std={pcg_out.std():.4f}, shape={pcg_out.shape}")